In [18]:
#create layout: https://pro.arcgis.com/en/pro-app/latest/arcpy/mapping/arcgisproject-class.htm
#map frame: https://pro.arcgis.com/en/pro-app/latest/arcpy/mapping/mapframe-class.htm#C_GUID-84CA2F83-ADE8-495F-96F5-A88E57E40F86
#legend: https://pro.arcgis.com/en/pro-app/latest/arcpy/mapping/legendelement-class.htm
import arcpy,os

aprx = arcpy.mp.ArcGISProject("CURRENT")
def OpenProject(aprx_route):
    #returns a dictionary with layers grouped by the map they're in.
    glob_layers={}
    nom_layers={}
    maps = aprx.listMaps()
    for map in maps:
        layers= map.listLayers()
        glob_layers[map.name]= layers
        map_layers=[]
        for layer in layers:
            map_layers.append(layer.name)
        nom_layers[map.name]= map_layers
    return glob_layers, nom_layers

def getGDB(input_gdb):
    gdb_list= []
    arcpy.env.workspace= input_gdb
    fc_list=  arcpy.ListFeatureClasses()
    rast_list= arcpy.ListRasters()
    if fc_list:
        for fc in fc_list: gdb_list.append(fc)
    if rast_list:
        for r in rast_list: gdb_list.append(r)
    return gdb_list


def createMapFrame(layout,mapf):
    lyt= layout
    #specify the lower left point as reference, and then the size of the map frame
    mf = lyt.createMapFrame(arcpy.Point(26.3948,92.3273), mapf, mapf.name)
    mf.elementWidth = 160.18
    mf.elementHeight = 159.1554 
    mf.locked= True
    mf.visible= False
    
def createLegend(layout):
    lyt= layout
    mf = lyt.listElements('MapFrame_Element')[0]
    legSi = aprx.listStyleItems('ArcGIS 2D', 'LEGEND','Legend 1' )[0]
    
    leg = lyt.createMapSurroundElement(arcpy.Point(14,280), 'LEGEND', mf,legSi,'Lleg')
    leg.elementWidth = 186
    leg.elementHeight = 270
    leg.fittingStrategy = 'AdjustColumns'
    leg.columnCount = 5
    leg.title = lyt.name
    leg.showTitle= True
    
def legItems(lyt):
    leg = lyt.listElements('LEGEND_ELEMENT')[0]
    
if __name__=='__main__':
    #first of all, list layer elements, whether they're in a gdb or if they're in a map
    layers= OpenProject(aprx)[1]['Map']
    g= getGDB(R'C:\Users\becari.alex.marcos\OneDrive - Institut Cartogràfic i Geològic de Catalunya\ALEX_MARCOS\RGT\geologia-territorial-25000\geologia-territorial-25000\geologia-territorial-25000-geologic.gdb')

    #now we mark the map we're going to use
    m= aprx.listMaps('Tek')[0]
    print(m)
    if False: #batch to create as many layouts as layers
        for ly in g:
            aprx.createLayout(210,297,'MILLIMETER',ly)
            #aprx.createLayout(page_width,page_height,units,name)
    if False: #create layout elements such as map frames or legends, automatically
        layouts= aprx.listLayouts()
        for lt in layouts:
            createMapFrame(lt,m)
            createLegend(lt)
            legItems(lt)